# $\mathcal{KUMARASWAMY}$ $\mathbf{Block\ GAPHC\ Project}$ — Final Core Implementation

**Scope:** Complete model formulation, simplified likelihood, robust unrestricted MLE with observed/expected Fisher Information, order-restricted MLE via PAVA with initial-value heuristics, and Bayesian MCMC estimation (unrestricted and order-restricted) with asymptotic variance estimation.

In [ ]:
import numpy as np
import scipy.optimize as opt
import scipy.stats as stats
import matplotlib.pyplot as plt

# Setup reproducibility and display settings
np.random.seed(42)
print("Libraries loaded successfully for Kumaraswamy Block GAPHC analysis.")

## 1. Standard Kumaraswamy Distribution

For $0 < x < 1$ and shape parameters $\alpha, \beta > 0$, the cumulative distribution function (CDF), probability density function (PDF), and survival function (SF) are defined as:

$$
F(x; \alpha, \beta) = 1 - \left(1 - x^{\alpha}\right)^{\beta}, \quad
f(x; \alpha, \beta) = \alpha \beta x^{\alpha-1} \left(1 - x^{\alpha}\right)^{\beta-1}
$$

$$
S(x; \alpha, \beta) = \left(1 - x^{\alpha}\right)^{\beta}
$$

In [ ]:
def kumaraswamy_pdf(x, alpha, beta):
    x = np.clip(x, 1e-12, 1.0 - 1e-12)
    return alpha * beta * (x ** (alpha - 1)) * ((1.0 - x ** alpha) ** (beta - 1))

def kumaraswamy_cdf(x, alpha, beta):
    x = np.clip(x, 0.0, 1.0)
    return 1.0 - (1.0 - x ** alpha) ** beta

def kumaraswamy_sf(x, alpha, beta):
    x = np.clip(x, 0.0, 1.0)
    return (1.0 - x ** alpha) ** beta

# Sanity check plot
x_vals = np.linspace(0.01, 0.99, 100)
plt.plot(x_vals, kumaraswamy_pdf(x_vals, alpha=2.0, beta=3.0), label=r'$\alpha=2, \beta=3$')
plt.title(r'Standard Kumaraswamy PDF $f(x; \alpha, \beta)$')
plt.xlabel(r'$x$')
plt.ylabel(r'$f(x)$')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

## 2. Block-Specific Working Model

For block $i = 1, \ldots, k$, we model the system with a common shape parameter $\alpha > 0$ and block-specific parameters $\lambda_i > 0$:

$$
f_i(x; \alpha, \lambda_i) = \alpha \lambda_i x^{\alpha-1} \left(1 - x^{\alpha}\right)^{\lambda_i-1},
\qquad
S_i(x; \alpha, \lambda_i) = \left(1 - x^{\alpha}\right)^{\lambda_i}
$$

For block $i$, let $n_i$ be initial units, $m_i$ observed failure times $x_{i1} < \cdots < x_{im_i}$, $R_{ij}$ progressive removals, $\tau_i$ terminal censoring time, and $C_i$ remaining censored units. The conservation identity is:

$$
n_i = m_i + \sum_{j=1}^{m_i} R_{ij} + C_i
$$

In [ ]:
def block_pdf(x, alpha, lambda_i):
    return kumaraswamy_pdf(x, alpha, lambda_i)

def block_sf(x, alpha, lambda_i):
    return kumaraswamy_sf(x, alpha, lambda_i)

def verify_count_identity(n_i, m_i, R_ij_array, C_i):
    sum_R = np.sum(R_ij_array)
    expected_n = m_i + sum_R + C_i
    assert n_i == expected_n, f"Count mismatch: n_i={n_i} vs m_i+sum(R)+C={expected_n}"
    return True

print("Block-specific model functions and verification helper defined.")

## 3. Likelihood & Transformed Data Functional

The full likelihood function $L(\boldsymbol{\theta})$ for parameter vector $\boldsymbol{\theta} = (\alpha, \lambda_1, \ldots, \lambda_k)^T$ under progressive hybrid censoring is:

$$
L(\boldsymbol{\theta}) = \prod_{i=1}^k \left[ \prod_{j=1}^{m_i} f_i(x_{ij}) \cdot S_i(x_{ij})^{R_{ij}} \right] S_i(\tau_i)^{C_i}
$$

Defining the data functional $W_i(\alpha)$:

$$
W_i(\alpha) = - \left[ \sum_{j=1}^{m_i} (1 + R_{ij}) \log\left(1 - x_{ij}^{\alpha}\right) + C_i \log\left(1 - \tau_i^{\alpha}\right) \right]
$$

The full log-likelihood function is:

$$
\ell(\boldsymbol{\theta}) = \sum_{i=1}^k \left[ m_i \log\alpha + m_i \log\lambda_i + (\alpha-1) \sum_{j=1}^{m_i} \log x_{ij} - \sum_{j=1}^{m_i} \log\left(1 - x_{ij}^{\alpha}\right) - \lambda_i W_i(\alpha) \right]
$$

In [ ]:
def compute_Wi(alpha, x_ij, R_ij, tau_i, C_i):
    x_ij = np.asarray(x_ij)
    R_ij = np.asarray(R_ij)
    term1 = np.sum((1.0 + R_ij) * np.log(1.0 - x_ij ** alpha))
    term2 = C_i * np.log(1.0 - tau_i ** alpha)
    return -(term1 + term2)

def log_likelihood(theta, data_dict):
    alpha = theta[0]
    lambdas = theta[1:]
    k = len(data_dict)
    ll = 0.0
    
    for i in range(k):
        d = data_dict[i]
        x_ij = np.array(d['x_ij'])
        R_ij = np.array(d['R_ij'])
        tau_i = d['tau_i']
        C_i = d['C_i']
        m_i = len(x_ij)
        lambda_i = lambdas[i]
        
        if alpha <= 0 or lambda_i <= 0 or np.any(x_ij <= 0) or np.any(x_ij >= 1) or tau_i >= 1:
            return -np.inf
            
        sum_log_x = np.sum(np.log(x_ij))
        sum_log_1_xmin = np.sum(np.log(1.0 - x_ij ** alpha))
        Wi = compute_Wi(alpha, x_ij, R_ij, tau_i, C_i)
        
        ll += (m_i * np.log(alpha) + m_i * np.log(lambda_i) +
               (alpha - 1.0) * sum_log_x + sum_log_1_xmin - lambda_i * Wi)
    return ll

print("Log-likelihood function implemented.")

## 4. Unrestricted MLE & Observed Fisher Information Matrix

Given $\alpha$, the profile estimator for $\lambda_i$ is:

$$
\widehat{\lambda}_i(\alpha) = \frac{m_i}{W_i(\alpha)}
$$

To compute asymptotic confidence intervals, we compute the Observed Fisher Information Matrix $\mathcal{I}(\boldsymbol{\theta}) = -\mathbf{H}(\boldsymbol{\theta})$, where $\mathbf{H}$ is the Hessian matrix of second partial derivatives evaluated at MLE $\widehat{\boldsymbol{\theta}}$:

$$
\operatorname{Var}(\widehat{\boldsymbol{\theta}}) \approx \mathcal{I}^{-1}(\widehat{\boldsymbol{\theta}})
$$

In [ ]:
def compute_initial_alpha(data_dict):
    """Heuristic starting value for shape parameter alpha using method of moments / log median."""
    all_x = []
    for d in data_dict.values():
        all_x.extend(d['x_ij'])
    all_x = np.array(all_x)
    med = np.median(all_x)
    # Initial rough alpha estimate based on median behavior
    alpha_init = -np.log(2) / np.log(med) if med > 0 and med < 1 else 1.0
    return max(0.1, float(alpha_init))

def profile_log_likelihood(alpha, data_dict):
    if alpha <= 0:
        return np.inf
    k = len(data_dict)
    lambdas_cond = np.zeros(k)
    for i in range(k):
        d = data_dict[i]
        x_ij = np.array(d['x_ij'])
        m_i = len(x_ij)
        if m_i == 0:
            return np.inf
        Wi = compute_Wi(alpha, x_ij, d['R_ij'], d['tau_i'], d['C_i'])
        if Wi <= 0:
            return np.inf
        lambdas_cond[i] = m_i / Wi
        
    theta = np.concatenate([[alpha], lambdas_cond])
    val = log_likelihood(theta, data_dict)
    return -val if np.isfinite(val) else np.inf

def fit_unrestricted_mle(data_dict):
    alpha_start = compute_initial_alpha(data_dict)
    res = opt.minimize_scalar(profile_log_likelihood, bracket=(alpha_start*0.5, alpha_start, alpha_start*2.0), args=(data_dict,), method='brent')
    alpha_mle = res.x
    k = len(data_dict)
    lambda_mles = np.zeros(k)
    for i in range(k):
        d = data_dict[i]
        m_i = len(d['x_ij'])
        Wi = compute_Wi(alpha_mle, d['x_ij'], d['R_ij'], d['tau_i'], d['C_i'])
        lambda_mles[i] = m_i / Wi
    return alpha_mle, lambda_mles

def compute_observed_fisher_info(theta_mle, data_dict, eps=1e-5):
    """Numerical Hessian to obtain Observed Fisher Information matrix and Standard Errors."""
    ndim = len(theta_mle)
    hessian = np.zeros((ndim, ndim))
    
    for i in range(ndim):
        for j in range(ndim):
            theta_ij_pp = np.copy(theta_mle)
            theta_ij_pm = np.copy(theta_mle)
            theta_ij_mp = np.copy(theta_mle)
            theta_ij_mm = np.copy(theta_mle)
            
            theta_ij_pp[i] += eps
            theta_ij_pp[j] += eps
            theta_ij_pm[i] += eps
            theta_ij_pm[j] -= eps
            theta_ij_mp[i] -= eps
            theta_ij_mp[j] += eps
            theta_ij_mm[i] -= eps
            theta_ij_mm[j] -= eps
            
            f_pp = log_likelihood(theta_ij_pp, data_dict)
            f_pm = log_likelihood(theta_ij_pm, data_dict)
            f_mp = log_likelihood(theta_ij_mp, data_dict)
            f_mm = log_likelihood(theta_ij_mm, data_dict)
            
            hessian[i, j] = (f_pp - f_pm - f_mp + f_mm) / (4 * eps**2)
            
    fisher_info = -hessian
    try:
        cov_matrix = np.linalg.inv(fisher_info)
        standard_errors = np.sqrt(np.maximum(0, np.diag(cov_matrix)))
    except np.linalg.LinAlgError:
        cov_matrix = None
        standard_errors = np.full(ndim, np.nan)
        
    return fisher_info, cov_matrix, standard_errors

print("Unrestricted MLE and Fisher Information routines defined.")

## 5. Order-Restricted MLE via PAVA

For the ordering restriction $0 < \lambda_1 \le \lambda_2 \le \cdots \le \lambda_k$, the Pool-Adjacent-Violators Algorithm (PAVA) computes exact isotonic estimations for fixed $\alpha$.

For boundary-constrained estimation, asymptotic variance matrix under order restrictions is evaluated using parameter-space projection or posterior variance via MCMC.

In [ ]:
def isotonic_regression_lambdas(alpha, data_dict):
    k = len(data_dict)
    m_vals = np.array([len(d['x_ij']) for d in data_dict])
    W_vals = np.array([compute_Wi(alpha, d['x_ij'], d['R_ij'], d['tau_i'], d['C_i']) for d in data_dict])
    
    lam = m_vals / W_vals
    blocks = [[i] for i in range(k)]
    weights = [W_vals[i] for i in range(k)]
    values = [lam[i] for i in range(k)]
    
    i = 0
    while i < len(blocks) - 1:
        if values[i] > values[i+1]:
            tot_m = sum(m_vals[idx] for idx in blocks[i]) + sum(m_vals[idx] for idx in blocks[i+1])
            tot_W = sum(W_vals[idx] for idx in blocks[i]) + sum(W_vals[idx] for idx in blocks[i+1])
            new_val = tot_m / tot_W
            blocks[i] = blocks[i] + blocks[i+1]
            values[i] = new_val
            weights[i] = tot_W
            blocks.pop(i+1)
            values.pop(i+1)
            weights.pop(i+1)
            if i > 0:
                i -= 1
        else:
            i += 1
            
    lam_restricted = np.zeros(k)
    for blk, val in zip(blocks, values):
        for idx in blk:
            lam_restricted[idx] = val
    return lam_restricted

def profile_log_likelihood_restricted(alpha, data_dict):
    if alpha <= 0:
        return np.inf
    lambdas_restricted = isotonic_regression_lambdas(alpha, data_dict)
    theta = np.concatenate([[alpha], lambdas_restricted])
    val = log_likelihood(theta, data_dict)
    return -val if np.isfinite(val) else np.inf

def fit_restricted_mle(data_dict):
    alpha_start = compute_initial_alpha(data_dict)
    res = opt.minimize_scalar(profile_log_likelihood_restricted, bracket=(alpha_start*0.5, alpha_start, alpha_start*2.0), args=(data_dict,), method='brent')
    alpha_mle = res.x
    lambda_mles = isotonic_regression_lambdas(alpha_mle, data_dict)
    return alpha_mle, lambda_mles

print("Order-restricted MLE estimation routine via PAVA defined.")

## 6. Bayesian Estimation: Prior & Posterior Setup

Independent Gamma priors are assigned:

$$
\alpha \sim \operatorname{Gamma}(a_0, b_0), \qquad
\lambda_i \sim \operatorname{Gamma}(a_i, b_i) \quad (i=1, \ldots, k)
$$

Posterior inference is obtained via Metropolis-Hastings MCMC sampling.

In [ ]:
def log_prior(theta, prior_params):
    alpha = theta[0]
    lambdas = theta[1:]
    a0, b0 = prior_params['alpha']
    
    if alpha <= 0 or np.any(lambdas <= 0):
        return -np.inf
        
    lp = stats.gamma.logpdf(alpha, a0, scale=1.0/b0)
    for i, lam in enumerate(lambdas):
        ai, bi = prior_params['lambda'][i]
        lp += stats.gamma.logpdf(lam, ai, scale=1.0/bi)
    return lp

def log_posterior_unrestricted(theta, data_dict, prior_params):
    ll = log_likelihood(theta, data_dict)
    lp = log_prior(theta, prior_params)
    return ll + lp if np.isfinite(ll) and np.isfinite(lp) else -np.inf

print("Prior and unrestricted posterior density functions defined.")

## 6.1 Unrestricted Bayesian MCMC Sampler

In [ ]:
def metropolis_hastings_sampler(data_dict, prior_params, init_theta, n_iter=5000, burn_in=1000, proposal_sd=0.05):
    ndim = len(init_theta)
    samples = np.zeros((n_iter, ndim))
    current_theta = np.array(init_theta, dtype=float)
    current_log_post = log_posterior_unrestricted(current_theta, data_dict, prior_params)
    
    accepted = 0
    for it in range(n_iter):
        proposal = current_theta + np.random.normal(0, proposal_sd, size=ndim)
        prop_log_post = log_posterior_unrestricted(proposal, data_dict, prior_params)
        
        log_accept_ratio = prop_log_post - current_log_post
        if np.log(np.random.uniform(0, 1)) < log_accept_ratio:
            current_theta = proposal
            current_log_post = prop_log_post
            if it >= burn_in:
                accepted += 1
                
        samples[it] = current_theta
        
    accepted_rate = accepted / (n_iter - burn_in)
    print(f"Unrestricted MCMC Acceptance Rate: {accepted_rate:.3f}")
    return samples[burn_in:]

print("Unrestricted Bayesian MCMC sampler defined.")

## 6.2 Order-Restricted Bayesian MCMC Sampler

Enforces the parameter domain $\Theta_R = \{ (\alpha, \lambda_1, \ldots, \lambda_k) : \alpha > 0, \ 0 < \lambda_1 \le \lambda_2 \le \cdots \le \lambda_k \}$ via truncated posterior sampling.

In [ ]:
def check_order_restriction(lambdas):
    return np.all(np.diff(lambdas) >= 0)

def log_posterior_restricted(theta, data_dict, prior_params):
    alpha = theta[0]
    lambdas = theta[1:]
    if not check_order_restriction(lambdas):
        return -np.inf
    return log_posterior_unrestricted(theta, data_dict, prior_params)

def metropolis_hastings_sampler_restricted(data_dict, prior_params, init_theta, n_iter=5000, burn_in=1000, proposal_sd=0.05):
    ndim = len(init_theta)
    samples = np.zeros((n_iter, ndim))
    current_theta = np.array(init_theta, dtype=float)
    
    if not check_order_restriction(current_theta[1:]):
        # Enforce initial order if violated
        current_theta[1:] = np.sort(current_theta[1:])
        
    current_log_post = log_posterior_restricted(current_theta, data_dict, prior_params)
    
    accepted = 0
    for it in range(n_iter):
        proposal = current_theta + np.random.normal(0, proposal_sd, size=ndim)
        prop_log_post = log_posterior_restricted(proposal, data_dict, prior_params)
        
        if np.isfinite(prop_log_post):
            log_accept_ratio = prop_log_post - current_log_post
            if np.log(np.random.uniform(0, 1)) < log_accept_ratio:
                current_theta = proposal
                current_log_post = prop_log_post
                if it >= burn_in:
                    accepted += 1
                    
        samples[it] = current_theta
        
    accepted_rate = accepted / (n_iter - burn_in)
    print(f"Restricted MCMC Acceptance Rate: {accepted_rate:.3f}")
    return samples[burn_in:]

print("Order-restricted Bayesian MCMC sampler defined.")

## 7. Numerical Verification & Execution Benchmark

In [ ]:
# Example demonstration with synthetic block GAPHC test data
synthetic_data = {
    0: {
        'x_ij': [0.15, 0.32, 0.54],
        'R_ij': [1, 0, 1],
        'tau_i': 0.85,
        'C_i': 1
    },
    1: {
        'x_ij': [0.22, 0.45, 0.61, 0.78],
        'R_ij': [0, 1, 0, 0],
        'tau_i': 0.90,
        'C_i': 0
    }
}

# Verify unit count identities
verify_count_identity(6, 3, [1, 0, 1], 1)
verify_count_identity(5, 4, [0, 1, 0, 0], 0)

# 1. Unrestricted MLE + Fisher Info
alpha_umle, lam_umle = fit_unrestricted_mle(synthetic_data)
theta_umle = np.concatenate([[alpha_umle], lam_umle])
fisher_info, cov_matrix, std_errors = compute_observed_fisher_info(theta_umle, synthetic_data)

print(f"Unrestricted MLE -> alpha: {alpha_umle:.4f}, lambdas: {lam_umle}")
print(f"Unrestricted Standard Errors: {std_errors}")

# 2. Order-Restricted MLE
alpha_rmle, lam_rmle = fit_restricted_mle(synthetic_data)
print(f"Order-Restricted MLE -> alpha: {alpha_rmle:.4f}, lambdas: {lam_rmle}")

# 3. Bayesian Samplers
prior_params = {
    'alpha': (2.0, 1.0),
    'lambda': [(2.0, 1.0), (2.0, 1.0)]
}
samples_unrest = metropolis_hastings_sampler(synthetic_data, prior_params, theta_umle, n_iter=2000, burn_in=500)
samples_rest = metropolis_hastings_sampler_restricted(synthetic_data, prior_params, theta_umle, n_iter=2000, burn_in=500)

print(f"Unrestricted Bayes Means: {np.mean(samples_unrest, axis=0)}")
print(f"Order-Restricted Bayes Means: {np.mean(samples_rest, axis=0)}")
print("All notebook functions tested successfully.")